In [1]:
"""
Module 4 — Active Learning Loop
=================================
Runs the active learning campaign:
    1. Start with 24 seed points (8 per library)
    2. Train heteroscedastic GP on observed points
    3. Compute movement-aware acquisition score for all pool points
    4. Select the point with highest score → reveal ground truth
    5. Add to observed set, repeat until pool is exhausted

Acquisition Function:
    score(x) = σ²_norm(x) - λ · C(x, x_current)

    Where C(x, x_current) is:
        within-library : d(x, x_current) / d_max        ∈ [0, 1]
        switch-library : 1 + γ                           > 1

    Guarantee: max within-library cost < min switch-library cost
    since max(d/d_max) = 1 < 1 + γ for any γ > 0

Evaluation:
    At every iteration, compute:
        - Global MAE for log_k0 and alpha
        - Per-library MAE for log_k0 and alpha
        - Number of library switches

Input:
    - seed_set.csv   (Module 2 output)
    - pool_set.csv   (Module 2 output)

Output:
    - al_results.csv      (MAE history per iteration)
    - al_sequence.csv     (order in which points were measured)
"""

'\nModule 4 — Active Learning Loop\n=================================\nRuns the active learning campaign:\n    1. Start with 24 seed points (8 per library)\n    2. Train heteroscedastic GP on observed points\n    3. Compute movement-aware acquisition score for all pool points\n    4. Select the point with highest score → reveal ground truth\n    5. Add to observed set, repeat until pool is exhausted\n\nAcquisition Function:\n    score(x) = σ²_norm(x) - λ · C(x, x_current)\n\n    Where C(x, x_current) is:\n        within-library : d(x, x_current) / d_max        ∈ [0, 1]\n        switch-library : 1 + γ                           > 1\n\n    Guarantee: max within-library cost < min switch-library cost\n    since max(d/d_max) = 1 < 1 + γ for any γ > 0\n\nEvaluation:\n    At every iteration, compute:\n        - Global MAE for log_k0 and alpha\n        - Per-library MAE for log_k0 and alpha\n        - Number of library switches\n\nInput:\n    - seed_set.csv   (Module 2 output)\n    - pool_set.

In [2]:
import warnings
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import gpflow
from pathlib import Path
# from tqdm import tqdm
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')



In [3]:
# Import Module 3 components
from _03_gpr_model import (
    Normaliser,
    build_heteroscedastic_svgp,
    train_model,
    predict,
    compute_d_max,
    INPUT_COLS,
    OUTPUT_COLS,
)

print("All imports successful")

All imports successful


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# ACQUISITION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

def compute_acquisition_scores(
    pool_df        : pd.DataFrame,
    log_k0_var     : np.ndarray,
    alpha_var      : np.ndarray,
    current_library: str,
    current_comp   : np.ndarray,    # ← just the type hint, no code here
    d_max          : float,
    lam            : float = 0.3,
    gamma          : float = 0.5,
) -> np.ndarray:
    """
    Compute movement-aware acquisition score for every pool point.

    Score = σ²_norm(x) - λ · C(x, x_current)

    Uncertainty term σ²_norm:
        Sum of normalised variances from both GP models.
        Normalised by current max variance so both outputs
        contribute equally regardless of their absolute scale.

    Movement cost C(x, x_current):
        Within-library : d(x, x_current) / d_max   ∈ [0, 1]
        Switch-library : 1 + γ                      > 1

        This guarantees the constraint:
            max within-library cost = 1 < 1 + γ = min switch cost
        So the robot never prefers switching over staying unless
        the uncertainty gain is large enough to overcome 1 + γ.

    Parameters
    ----------
    pool_df         : current pool DataFrame
    log_k0_var      : GP variance for log_k0, shape (n_pool,)
    alpha_var       : GP variance for alpha,  shape (n_pool,)
    current_library : library of the last measured point
    current_comp    : [Au%, Ir%] of the last measured point
    d_max           : maximum composition distance across all points
    lam             : penalty weight λ (0 = pure uncertainty, 1 = max penalty)
    gamma           : switch library extra cost γ (must be > 0)

    Returns
    -------
    scores : acquisition scores, shape (n_pool,)
             Higher = more valuable to measure next
    """
    # ── Force correct types at the start of the function ─────────────────
    current_comp = np.array(current_comp, dtype=np.float64).flatten()
    pool_comps   = pool_df[INPUT_COLS].values.astype(np.float64)
    libraries    = pool_df['library'].values

    # ── Uncertainty term ──────────────────────────────────────────────────
    log_k0_var_norm = log_k0_var / (log_k0_var.max() + 1e-10)
    alpha_var_norm  = alpha_var  / (alpha_var.max()  + 1e-10)
    uncertainty     = log_k0_var_norm + alpha_var_norm

    # ── Movement cost ─────────────────────────────────────────────────────
    diffs          = pool_comps - current_comp
    distances      = np.sqrt((diffs ** 2).sum(axis=1))
    distances_norm = distances / (d_max + 1e-10)

    same_library  = (libraries == current_library)
    movement_cost = np.where(
        same_library,
        distances_norm,
        1.0 + gamma,
    )

    # ── Final score ───────────────────────────────────────────────────────
    scores = uncertainty - lam * movement_cost
    return scores


# ─────────────────────────────────────────────────────────────────────────────
# MAE EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_mae(
    pool_df   : pd.DataFrame,
    log_k0_mu : np.ndarray,
    alpha_mu  : np.ndarray,
) -> dict:
    """
    Compute normalised MAE for log_k0 and alpha on the pool set.

    Normalised MAE = MAE / (max - min) of ground truth
    This matches the metric used in the original paper and
    makes results comparable across outputs with different scales.

    Returns
    -------
    dict with keys:
        log_k0_mae_global, alpha_mae_global   (float)
        log_k0_mae_<library>, alpha_mae_<library>  (float per library)
    """
    results = {}

    true_log_k0 = pool_df['log_k0'].values
    true_alpha  = pool_df['alpha'].values

    # Normalisation ranges — computed from ground truth
    log_k0_range = true_log_k0.max() - true_log_k0.min() + 1e-10
    alpha_range  = true_alpha.max()  - true_alpha.min()  + 1e-10

    # Global MAE
    results['log_k0_mae_global'] = np.abs(true_log_k0 - log_k0_mu).mean() / log_k0_range
    results['alpha_mae_global']  = np.abs(true_alpha   - alpha_mu).mean()  / alpha_range

    # Per-library MAE
    for library in pool_df['library'].unique():
        mask = pool_df['library'].values == library
        if mask.sum() == 0:
            continue
        results[f'log_k0_mae_{library}'] = (
            np.abs(true_log_k0[mask] - log_k0_mu[mask]).mean() / log_k0_range
        )
        results[f'alpha_mae_{library}'] = (
            np.abs(true_alpha[mask] - alpha_mu[mask]).mean() / alpha_range
        )

    return results


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# SINGLE ITERATION — retrain + predict + score + select
# ─────────────────────────────────────────────────────────────────────────────

def run_iteration(
    observed_df    : pd.DataFrame,
    pool_df        : pd.DataFrame,
    normaliser     : Normaliser,
    d_max          : float,
    current_library: str,
    current_comp   : np.ndarray,
    n_inducing     : int   = 20,
    max_iter       : int   = 300,
    lam            : float = 0.3,
    gamma          : float = 0.5,
) -> tuple[dict, int, gpflow.models.SVGP, gpflow.models.SVGP, np.ndarray, np.ndarray]:
    """
    Run one active learning iteration:
        1. Retrain GP on all observed points
        2. Predict on pool
        3. Evaluate MAE
        4. Compute acquisition scores
        5. Return index of best next point

    Returns
    -------
    mae_dict        : MAE metrics for this iteration
    next_idx        : index in pool_df of the next point to measure
    model_log_k0    : trained GP for log_k0
    model_alpha     : trained GP for alpha
    log_k0_mu       : predicted means for log_k0
    alpha_mu        : predicted means for alpha
    """
    # ── Prepare arrays ────────────────────────────────────────────────────────
    X_obs  = normaliser.transform(observed_df[INPUT_COLS].values.astype(np.float64))
    X_pool = normaliser.transform(pool_df[INPUT_COLS].values.astype(np.float64))

    # ── Train and predict for each output ─────────────────────────────────────
    predictions = {}
    models      = {}

    for output_col in OUTPUT_COLS:
        Y_obs = observed_df[output_col].values.reshape(-1, 1).astype(np.float64)

        model = build_heteroscedastic_svgp(X_obs, n_inducing=min(n_inducing, len(X_obs)))
        train_model(model, X_obs, Y_obs, max_iter=max_iter, verbose=False)
        mu, var = predict(model, X_pool)

        models[output_col]               = model
        predictions[f'{output_col}_mu']  = mu
        predictions[f'{output_col}_var'] = var

    # ── Evaluate MAE ──────────────────────────────────────────────────────────
    mae_dict = evaluate_mae(
        pool_df,
        predictions['log_k0_mu'],
        predictions['alpha_mu'],
    )

    # ── Acquisition scores ────────────────────────────────────────────────────
    scores = compute_acquisition_scores(
        pool_df         = pool_df,
        log_k0_var      = predictions['log_k0_var'],
        alpha_var       = predictions['alpha_var'],
        current_library = current_library,
        current_comp    = current_comp,
        d_max           = d_max,
        lam             = lam,
        gamma           = gamma,
    )

    # Best next point = highest acquisition score
    next_idx = int(np.argmax(scores))

    return (
        mae_dict,
        next_idx,
        models['log_k0'],
        models['alpha'],
        predictions['log_k0_mu'],
        predictions['alpha_mu'],
    )

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# FULL ACTIVE LEARNING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def run_active_learning(
    seed_path  : str,
    pool_path  : str,
    output_dir : str   = "data",
    n_inducing : int   = 20,
    max_iter   : int   = 300,
    lam        : float = 0.3,
    gamma      : float = 0.5,
    eval_every : int   = 1,
) -> pd.DataFrame:
    """
    Run the full active learning loop from seed set to exhausted pool.

    Parameters
    ----------
    seed_path  : path to seed_set.csv
    pool_path  : path to pool_set.csv
    output_dir : folder to save results
    n_inducing : SVGP inducing points
    max_iter   : GP optimisation iterations per loop
    lam        : movement penalty weight λ
    gamma      : switch-library extra penalty γ
    eval_every : evaluate MAE every N iterations (1 = every step)

    Returns
    -------
    results_df : DataFrame with MAE history across all iterations
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Load data ─────────────────────────────────────────────────────────────
    print("=" * 55)
    print("LOADING DATA")
    print("=" * 55)
    seed_df = pd.read_csv(seed_path)
    pool_df = pd.read_csv(pool_path).reset_index(drop=True)
    print(f"  Seed set : {len(seed_df)} points")
    print(f"  Pool set : {len(pool_df)} points")

    # ── Compute d_max ─────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("COMPUTING d_max")
    print("=" * 55)
    d_max = compute_d_max(seed_df, pool_df)

    # ── Fit normaliser on seed set ────────────────────────────────────────────
    normaliser = Normaliser()
    normaliser.fit(seed_df[INPUT_COLS].values.astype(np.float64))

    # ── Initialise observed set and pool ─────────────────────────────────────
    # observed_df grows as points are revealed
    # pool_df shrinks as points are selected
    observed_df = seed_df.copy()
    remaining_pool = pool_df.copy()

    # Starting position — centroid of seed set (no prior measurement)
    current_library = seed_df['library'].mode()[0]
    current_comp    = seed_df[INPUT_COLS].values.mean(axis=0).astype(np.float64)


    # ── Tracking variables ────────────────────────────────────────────────────
    results       = []        # MAE history
    sequence      = []        # order of measurements
    n_switches    = 0         # library switch counter
    total_iters   = len(pool_df)

    print("\n" + "=" * 55)
    print(f"STARTING ACTIVE LEARNING LOOP")
    print(f"  Total iterations : {total_iters}")
    print(f"  λ (penalty weight)   : {lam}")
    print(f"  γ (switch penalty)   : {gamma}")
    print("=" * 55)

    # ── Main loop ─────────────────────────────────────────────────────────────
    for iteration in range(total_iters):

        n_observed = len(observed_df)

        # Only evaluate MAE every eval_every steps to save time
        # Always evaluate at the first and last iteration
        do_eval = (iteration % eval_every == 0) or (iteration == total_iters - 1)

        mae_dict, next_idx, _, _, log_k0_mu, alpha_mu = run_iteration(
            observed_df     = observed_df,
            pool_df         = remaining_pool,
            normaliser      = normaliser,
            d_max           = d_max,
            current_library = current_library,
            current_comp    = current_comp,
            n_inducing      = n_inducing,
            max_iter        = max_iter,
            lam             = lam,
            gamma           = gamma,
        )

        # ── Record MAE ────────────────────────────────────────────────────────
        if do_eval:
            row = {'iteration': iteration, 'n_observed': n_observed}
            row.update(mae_dict)
            results.append(row)

            print(
                f"  Iter {iteration+1:4d}/{total_iters} | "
                f"n_obs={n_observed:4d} | "
                f"log_k0 MAE={mae_dict['log_k0_mae_global']*100:.2f}% | "
                f"alpha MAE={mae_dict['alpha_mae_global']*100:.2f}% | "
                f"switches={n_switches}"
            )

        # ── Select next point ─────────────────────────────────────────────────
        next_point = remaining_pool.iloc[next_idx].copy()
        next_library = next_point['library']
        next_comp    = next_point[INPUT_COLS].values.astype(np.float64)


        # Count library switches
        if next_library != current_library:
            n_switches += 1

        # Record the sequence of measurements
        sequence.append({
            'iteration'     : iteration,
            'library'       : next_library,
            'area'          : next_point['area'],
            'Au [at.%]'     : next_point['Au [at.%]'],
            'Ir [at.%]'     : next_point['Ir [at.%]'],
            'true_log_k0'   : next_point['log_k0'],
            'true_alpha'    : next_point['alpha'],
            'library_switch': int(next_library != current_library),
        })

        # ── Update state ──────────────────────────────────────────────────────
        # Add selected point to observed set
        observed_df = pd.concat(
            [observed_df, next_point.to_frame().T],
            ignore_index=True
        )
        # Remove from pool — drop by positional index
        remaining_pool = remaining_pool.drop(
            index=remaining_pool.index[next_idx]
        ).reset_index(drop=True)

        # Update current position
        current_library = next_library
        current_comp    = next_comp

    # ── Save results ──────────────────────────────────────────────────────────
    results_df  = pd.DataFrame(results)
    sequence_df = pd.DataFrame(sequence)

    results_path  = output_dir / "al_results.csv"
    sequence_path = output_dir / "al_sequence.csv"

    results_df.to_csv(results_path,   index=False)
    sequence_df.to_csv(sequence_path, index=False)

    print("\n" + "=" * 55)
    print("DONE")
    print("=" * 55)
    print(f"  MAE history saved to   : {results_path}")
    print(f"  Measurement sequence   : {sequence_path}")
    print(f"  Total library switches : {n_switches} / {total_iters} iterations")
    print(f"  Final log_k0 MAE       : {results_df['log_k0_mae_global'].iloc[-1]*100:.2f}%")
    print(f"  Final alpha MAE        : {results_df['alpha_mae_global'].iloc[-1]*100:.2f}%")

    return results_df

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────


print("\n" + "=" * 55)
print("MODULE 4 — ACTIVE LEARNING LOOP")
print("=" * 55)

seed_path = input("\nPath to seed_set.csv:\n> ").strip().strip('"').strip("'")
pool_path = input("\nPath to pool_set.csv:\n> ").strip().strip('"').strip("'")

lam_in = input("\nMovement penalty weight λ [default: 0.3]:\n> ").strip()
lam    = float(lam_in) if lam_in else 0.3

gam_in = input("\nSwitch-library extra penalty γ [default: 0.5]:\n> ").strip()
gamma  = float(gam_in) if gam_in else 0.5

n_ind  = input("\nNumber of inducing points [default: 20]:\n> ").strip()
n_inducing = int(n_ind) if n_ind else 20

iters  = input("\nMax GP optimisation iterations [default: 300]:\n> ").strip()
max_iter = int(iters) if iters else 300

every  = input("\nEvaluate MAE every N iterations [default: 5]:\n> ").strip()
eval_every = int(every) if every else 5

out_dir = input("\nFolder to save results [default: data]:\n> ").strip().strip('"').strip("'")
output_dir = out_dir if out_dir else "data"

results = run_active_learning(
    seed_path  = seed_path,
    pool_path  = pool_path,
    output_dir = output_dir,
    n_inducing = n_inducing,
    max_iter   = max_iter,
    lam        = lam,
    gamma      = gamma,
    eval_every = eval_every,
)


MODULE 4 — ACTIVE LEARNING LOOP



Path to seed_set.csv:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\seed_set.csv

Path to pool_set.csv:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\pool_set.csv

Movement penalty weight λ [default: 0.3]:
>  

Switch-library extra penalty γ [default: 0.5]:
>  

Number of inducing points [default: 20]:
>  120

Max GP optimisation iterations [default: 300]:
>  

Evaluate MAE every N iterations [default: 5]:
>  

Folder to save results [default: data]:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\testing\final


LOADING DATA
  Seed set : 24 points
  Pool set : 942 points

COMPUTING d_max
  d_max (max composition distance): 116.7728 at.% units

STARTING ACTIVE LEARNING LOOP
  Total iterations : 942
  λ (penalty weight)   : 0.3
  γ (switch penalty)   : 0.5
  Iter    1/942 | n_obs=  24 | log_k0 MAE=8.71% | alpha MAE=14.49% | switches=0
  Iter    6/942 | n_obs=  29 | log_k0 MAE=8.47% | alpha MAE=13.43% | switches=1
  Iter   11/942 | n_obs=  34 | log_k0 MAE=9.01% | alpha MAE=12.84% | switches=1
  Iter   16/942 | n_obs=  39 | log_k0 MAE=8.38% | alpha MAE=12.59% | switches=2
  Iter   21/942 | n_obs=  44 | log_k0 MAE=7.70% | alpha MAE=12.59% | switches=3
  Iter   26/942 | n_obs=  49 | log_k0 MAE=7.42% | alpha MAE=12.33% | switches=4
  Iter   31/942 | n_obs=  54 | log_k0 MAE=7.30% | alpha MAE=12.41% | switches=4
  Iter   36/942 | n_obs=  59 | log_k0 MAE=7.34% | alpha MAE=12.46% | switches=4
  Iter   41/942 | n_obs=  64 | log_k0 MAE=7.03% | alpha MAE=12.21% | switches=5
  Iter   46/942 | n_obs=  69 | lo